# EDA 02 — OpenStreetMap : Points d'Interet Parisiens
**Source** : API Overpass | **Bronze** : `data/bronze/osm/`

In [ ]:

import os, glob, warnings
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import numpy as np
warnings.filterwarnings("ignore")

PALETTE = {
    "primary":   "#1565C0",
    "secondary": "#E53935",
    "accent":    "#2E7D32",
    "neutral":   "#546E7A",
}

plt.rcParams.update({
    "figure.dpi": 150,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "axes.grid.axis": "y",
    "grid.alpha": 0.3,
    "grid.linestyle": "--",
    "font.family": "sans-serif",
    "font.size": 11,
    "axes.titlesize": 14,
    "axes.titleweight": "bold",
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.framealpha": 0.9,
    "legend.fontsize": 10,
})

BRONZE_ROOT = os.path.abspath("../../data/bronze")
NB_DIR      = os.path.abspath(".")
FIGURES_DIR = os.path.join(NB_DIR, "figures")
os.makedirs(FIGURES_DIR, exist_ok=True)

def save_fig(name, source_prefix, tight=True):
    dest = os.path.join(FIGURES_DIR, source_prefix)
    os.makedirs(dest, exist_ok=True)
    path = os.path.join(dest, f"{name}.png")
    if tight:
        plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches="tight", facecolor="white")
    print(f"  Saved -> figures/{source_prefix}/{name}.png")

print("Setup OK — figures ->", FIGURES_DIR)


In [ ]:

def draw_schema(title, columns, source_prefix, filename="schema"):
    n_rows = len(columns)
    fig_h  = max(3.0, 0.48 * n_rows + 1.6)
    fig, ax = plt.subplots(figsize=(13, fig_h))
    ax.axis("off")
    fig.suptitle(title, fontsize=14, fontweight="bold", y=0.99, ha="center")
    headers = ["Colonne", "Type", "Description"]
    col_x   = [0.0, 0.22, 0.37]
    col_w   = [0.22, 0.15, 0.63]
    row_h   = 1.0 / (n_rows + 1)
    for i, (hdr, x, w) in enumerate(zip(headers, col_x, col_w)):
        rect = mpatches.FancyBboxPatch(
            (x, 1 - row_h), w - 0.006, row_h * 0.88,
            boxstyle="round,pad=0.01", linewidth=0.5,
            edgecolor="#90A4AE", facecolor="#1565C0",
            transform=ax.transAxes, clip_on=False)
        ax.add_patch(rect)
        ax.text(x + w/2, 1 - row_h/2, hdr,
                transform=ax.transAxes, ha="center", va="center",
                fontsize=11, fontweight="bold", color="white")
    type_colors = {
        "str":"#1565C0","int":"#2E7D32","float":"#6A1B9A",
        "datetime":"#E65100","bool":"#AD1457","date":"#00695C",
    }
    for ridx, (col_name, col_type, col_desc) in enumerate(columns):
        y_top = 1 - (ridx + 2) * row_h
        bg = "#F5F7FA" if ridx % 2 == 0 else "white"
        for x, w in zip(col_x, col_w):
            rect = mpatches.FancyBboxPatch(
                (x, y_top), w - 0.006, row_h * 0.88,
                boxstyle="round,pad=0.01", linewidth=0.5,
                edgecolor="#CFD8DC", facecolor=bg,
                transform=ax.transAxes, clip_on=False)
            ax.add_patch(rect)
        y_mid = y_top + row_h * 0.44
        tc = type_colors.get(col_type.split("[")[0].lower(), "#37474F")
        ax.text(col_x[0]+col_w[0]/2, y_mid, col_name,
                transform=ax.transAxes, ha="center", va="center",
                fontsize=10, fontweight="bold", color="#1A237E")
        ax.text(col_x[1]+col_w[1]/2, y_mid, col_type,
                transform=ax.transAxes, ha="center", va="center",
                fontsize=9, fontfamily="monospace", color=tc)
        ax.text(col_x[2]+0.01, y_mid, col_desc,
                transform=ax.transAxes, ha="left", va="center",
                fontsize=9.5, color="#37474F")
    plt.subplots_adjust(top=0.92, bottom=0)
    save_fig(filename, source_prefix)
    plt.show()


In [ ]:
PREFIX = "02_osm"
draw_schema(
    "Bronze Schema — OSM Points d'Interet (OpenStreetMap)",
    [
        ("osm_id",        "int",      "Identifiant unique OSM"),
        ("osm_type",      "str",      "Type d'element : node ou way"),
        ("amenity_type",  "str",      "Categorie : bar | cinema | nightclub | park | restaurant | school | shop | stadium | university"),
        ("name",          "str",      "Nom du lieu (nullable)"),
        ("latitude",      "float",    "Latitude (centroide pour les ways)"),
        ("longitude",     "float",    "Longitude (centroide pour les ways)"),
        ("opening_hours", "str",      "Horaires OSM (nullable)"),
        ("wheelchair",    "str",      "Accessibilite PMR (nullable)"),
        ("tags",          "str",      "Dictionnaire complet des tags OSM (JSON encode)"),
        ("ingested_at",   "datetime", "Horodatage UTC d'ingestion"),
    ], PREFIX
)

In [ ]:
osm_files = sorted(glob.glob(f"{BRONZE_ROOT}/osm/**/*.parquet", recursive=True))
if osm_files:
    df = pd.concat([pd.read_parquet(f) for f in osm_files], ignore_index=True)
else:
    rng = np.random.default_rng(42)
    cfg = {"bar":900,"cinema":45,"nightclub":280,"park":210,"restaurant":2800,"school":400,"shop":650}
    dfs = []
    for atype, n in cfg.items():
        arr = rng.integers(1,21,n)
        dfs.append(pd.DataFrame({"osm_id":rng.integers(int(1e6),int(9e6),n),"osm_type":rng.choice(["node","way"],n),
            "amenity_type":atype,"name":[f"{atype} {i}" for i in range(n)],
            "latitude":48.8566+rng.uniform(-0.08,0.08,n),"longitude":2.3522+rng.uniform(-0.09,0.09,n),"arrondissement":arr}))
    df = pd.concat(dfs,ignore_index=True)
print(f"Shape : {df.shape}")
df["amenity_type"].value_counts()

In [ ]:
type_counts = df["amenity_type"].value_counts().sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(11,6))
bar_colors = [plt.cm.Blues(0.3+i/len(type_counts)*0.6) for i in range(len(type_counts))]
bars = ax.barh(type_counts.index, type_counts.values, color=bar_colors, edgecolor="white", height=0.65)
for bar, val in zip(bars, type_counts.values):
    ax.text(val+15, bar.get_y()+bar.get_height()/2, f"{val:,}", va="center", fontsize=10, color=PALETTE["neutral"])
ax.set_xlabel("Nombre de POI"); ax.set_title("Repartition des POI OpenStreetMap par categorie — Paris"); ax.grid(axis="x")
save_fig("repartition_categories", PREFIX)
plt.show()
print(f"Total POI : {len(df):,}")

In [ ]:
cat_colors = {"bar":"#E53935","nightclub":"#8E24AA","park":"#2E7D32",
    "restaurant":"#FF8F00","school":"#1565C0","shop":"#00838F","cinema":"#6D4C41"}
fig, axes = plt.subplots(1,2,figsize=(16,7))
for atype, grp in df.groupby("amenity_type"):
    axes[0].scatter(grp["longitude"],grp["latitude"],s=2,alpha=0.5,color=cat_colors.get(atype,"#999"),label=atype)
axes[0].set_title("Distribution geographique des POI"); axes[0].set_xlabel("Longitude"); axes[0].set_ylabel("Latitude")
axes[0].legend(markerscale=6,fontsize=8,loc="lower right",ncol=2); axes[0].set_aspect("equal")
focus = df[df["amenity_type"].isin(["bar","restaurant","park","school"])]
for atype, grp in focus.groupby("amenity_type"):
    axes[1].scatter(grp["longitude"],grp["latitude"],s=3,alpha=0.6,color=cat_colors.get(atype,"#999"),label=atype)
axes[1].set_title("Focus : bars, restaurants, parcs, ecoles"); axes[1].legend(markerscale=5,fontsize=9); axes[1].set_aspect("equal")
save_fig("distribution_geographique", PREFIX)
plt.show()